# HP_NB_DeltaOptimization

## Purpose

This notebook demonstrates Delta Lake maintenance and performance optimization for the HealthPulse Silver Lakehouse.

It focuses on:

- checking Delta table health
- measuring a simple baseline query
- compacting small files with `OPTIMIZE`
- applying V-Order for read-heavy analytics
- previewing or running `VACUUM`
- reviewing partition strategy
- checking schema and Delta history
- recording before/after results

This notebook **does not** perform Bronze-to-Silver transformations.


## 1. Imports and configuration

For the portfolio demonstration, `clinical.Encounter` is the main table. The same pattern can later be reused for other large Silver tables.


In [1]:
import time
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F

SILVER_LAKEHOUSE = "HP_02_LH_Health_Silver"
TARGET_SCHEMA = "clinical"
TARGET_TABLE_NAME = "Encounter"
TARGET_TABLE = f"{SILVER_LAKEHOUSE}.{TARGET_SCHEMA}.{TARGET_TABLE_NAME}"

# Other tables that may need maintenance later.
TABLES_TO_OPTIMIZE = [
    "clinical.Encounter",
    "clinical.Observation",
    "clinical.Condition",
    "clinical.Procedure",
    "clinical.Medication"
]

# Use V-Order for this read-heavy demonstration.
APPLY_VORDER = True

# VACUUM is disabled by default for safety.
# A DRY RUN can be used to preview old files that are eligible for removal.
RUN_VACUUM_DRY_RUN = True
RUN_VACUUM = False
VACUUM_RETENTION_HOURS = 168   # 7 days

optimization_run_id = str(uuid.uuid4())
optimization_start_time = datetime.now(timezone.utc).replace(tzinfo=None)

print("Optimization Run ID:", optimization_run_id)
print("Target table       :", TARGET_TABLE)


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 3, Finished, Available, Finished, False)

Optimization Run ID: 4e79c87d-fcf5-4a8a-a3f7-97d261220659
Target table       : HP_02_LH_Health_Silver.clinical.Encounter


## 2. Validate the target table

Fail early if the Silver Encounter table is not available.


In [2]:
try:
    spark.read.table(TARGET_TABLE).limit(1).count()
    print(f"Table found: {TARGET_TABLE}")
except Exception as e:
    raise ValueError(
        f"The target table could not be read: {TARGET_TABLE}"
    ) from e


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 4, Finished, Available, Finished, False)

Table found: HP_02_LH_Health_Silver.clinical.Encounter


## 3. Create the optimization log

The log keeps a simple history of optimization runs so the results can later be shown in the Engineering Operations dashboard.


In [3]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {SILVER_LAKEHOUSE}.ops
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_LAKEHOUSE}.ops.DeltaOptimizationRun
(
    OptimizationRunID STRING,
    TableName STRING,
    StartTime TIMESTAMP,
    EndTime TIMESTAMP,
    FilesBefore BIGINT,
    FilesAfter BIGINT,
    SizeBeforeBytes BIGINT,
    SizeAfterBytes BIGINT,
    QueryTimeBeforeSeconds DOUBLE,
    QueryTimeAfterSeconds DOUBLE,
    OptimizeApplied BOOLEAN,
    VOrderApplied BOOLEAN,
    VacuumApplied BOOLEAN,
    Status STRING,
    Notes STRING
)
USING DELTA
""")

print("Optimization log table is ready.")


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 5, Finished, Available, Finished, False)

Optimization log table is ready.


## 4. Helper functions

`get_table_stats()` reads basic Delta statistics.

`run_test_query()` runs the same analytical query before and after optimization.


In [4]:
def get_table_stats(table_name):
    """Return simple Delta table statistics."""

    detail = spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    ).first().asDict()

    num_files = int(detail.get("numFiles") or 0)
    size_bytes = int(detail.get("sizeInBytes") or 0)

    return {
        "numFiles": num_files,
        "sizeInBytes": size_bytes,
        "averageFileSizeBytes": (
            size_bytes / num_files if num_files > 0 else 0
        ),
        "partitionColumns": detail.get("partitionColumns") or []
    }


def run_test_query():
    """Run one simple Encounter aggregation and return elapsed seconds."""

    # Clear Spark cache so the comparison is a little fairer.
    spark.catalog.clearCache()

    start = time.perf_counter()

    result = spark.sql(f"""
        SELECT
            YEAR(EncounterDate) AS EncounterYear,
            MONTH(EncounterDate) AS EncounterMonth,
            COUNT(*) AS EncounterCount
        FROM {TARGET_TABLE}
        GROUP BY
            YEAR(EncounterDate),
            MONTH(EncounterDate)
        ORDER BY
            EncounterYear,
            EncounterMonth
    """)

    result.collect()

    return time.perf_counter() - start


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 6, Finished, Available, Finished, False)

## 5. Inspect the table before optimization

The most useful first indicators are the number of files, active table size, average file size, and current partition columns.


In [5]:
before_stats = get_table_stats(TARGET_TABLE)

print("Before optimization")
print("-------------------")
print("Files              :", before_stats["numFiles"])
print("Size in bytes      :", before_stats["sizeInBytes"])
print(
    "Average file MB    :",
    round(before_stats["averageFileSizeBytes"] / (1024 * 1024), 2)
)
print("Partition columns  :", before_stats["partitionColumns"])


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 7, Finished, Available, Finished, False)

Before optimization
-------------------
Files              : 10
Size in bytes      : 1080565700
Average file MB    : 103.05
Partition columns  : []


## 6. Measure query performance before optimization

This is a simple portfolio benchmark. For a formal benchmark, run the same query several times under comparable conditions.


In [6]:
query_time_before = run_test_query()

print(
    f"Query time before optimization: "
    f"{query_time_before:.3f} seconds"
)


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 8, Finished, Available, Finished, False)

Query time before optimization: 4.079 seconds


## 7. Run OPTIMIZE

`OPTIMIZE` compacts small Delta files. If `APPLY_VORDER` is `True`, V-Order is applied in the same operation, so we do not run OPTIMIZE twice.


In [7]:
optimize_applied = False
vorder_applied = False
vacuum_applied = False
notes = []

if APPLY_VORDER:
    print("Running OPTIMIZE with V-Order...")
    spark.sql(f"OPTIMIZE {TARGET_TABLE} VORDER")
    optimize_applied = True
    vorder_applied = True
    notes.append("OPTIMIZE with V-Order executed")
else:
    print("Running OPTIMIZE...")
    spark.sql(f"OPTIMIZE {TARGET_TABLE}")
    optimize_applied = True
    notes.append("OPTIMIZE executed")

print("Optimization completed.")


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 9, Finished, Available, Finished, False)

Running OPTIMIZE with V-Order...
Optimization completed.


## 8. Measure the table after optimization

Use the same table statistics and the same query so the comparison is easy to explain.


In [8]:
after_stats = get_table_stats(TARGET_TABLE)
query_time_after = run_test_query()

improvement_pct = (
    ((query_time_before - query_time_after) / query_time_before) * 100
    if query_time_before > 0
    else 0.0
)

print("After optimization")
print("------------------")
print("Files              :", after_stats["numFiles"])
print("Size in bytes      :", after_stats["sizeInBytes"])
print(
    "Average file MB    :",
    round(after_stats["averageFileSizeBytes"] / (1024 * 1024), 2)
)
print("Partition columns  :", after_stats["partitionColumns"])
print(f"Query time          : {query_time_after:.3f} seconds")
print(f"Measured change     : {improvement_pct:.2f}%")


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 10, Finished, Available, Finished, False)

After optimization
------------------
Files              : 10
Size in bytes      : 1080565700
Average file MB    : 103.05
Partition columns  : []
Query time          : 1.571 seconds
Measured change     : 61.48%


## 9. VACUUM

`VACUUM` removes obsolete files after the retention period. It is mainly a storage-cleanup operation.

For safety:

- the default retention here is 168 hours (7 days)
- a `DRY RUN` is enabled by default
- the real VACUUM is disabled until `RUN_VACUUM = True`

Do not shorten retention simply for a demonstration because older files may be needed for time travel or recovery.


In [9]:
if RUN_VACUUM_DRY_RUN:
    print(
        f"VACUUM DRY RUN with retention "
        f"{VACUUM_RETENTION_HOURS} hours"
    )

    vacuum_preview = spark.sql(f"""
        VACUUM {TARGET_TABLE}
        RETAIN {VACUUM_RETENTION_HOURS} HOURS
        DRY RUN
    """)

    display(vacuum_preview)
    notes.append("VACUUM DRY RUN executed")

if RUN_VACUUM:
    print(
        f"Running VACUUM with retention "
        f"{VACUUM_RETENTION_HOURS} hours..."
    )

    spark.sql(f"""
        VACUUM {TARGET_TABLE}
        RETAIN {VACUUM_RETENTION_HOURS} HOURS
    """)

    vacuum_applied = True
    notes.append(
        f"VACUUM executed with {VACUUM_RETENTION_HOURS}-hour retention"
    )

if not RUN_VACUUM:
    print("Actual VACUUM was not run. Change RUN_VACUUM to True when appropriate.")


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 11, Finished, Available, Finished, False)

VACUUM DRY RUN with retention 168 hours


SynapseWidget(Synapse.DataFrame, acdb2c7c-f363-4797-bc60-3f3eca0a257a)

Actual VACUUM was not run. Change RUN_VACUUM to True when appropriate.


## 10. Review partition strategy

HealthPulse does not partition every table.

For Encounter, time-based columns such as year/month are reasonable candidates only when the table becomes large enough and query patterns justify partitioning.

`PatientID` should not be used as a partition column because it has very high cardinality and can create too many small partitions.


In [10]:
encounter_df = spark.table(TARGET_TABLE)

if "EncounterDate" in encounter_df.columns:
    encounter_by_month = (
        encounter_df
        .groupBy(
            F.year("EncounterDate").alias("EncounterYear"),
            F.month("EncounterDate").alias("EncounterMonth")
        )
        .count()
        .orderBy("EncounterYear", "EncounterMonth")
    )

    display(encounter_by_month)
else:
    print("EncounterDate was not found; time-based partition review was skipped.")

print("Current physical partition columns:", after_stats["partitionColumns"])
print("No repartitioning is performed automatically in this notebook.")


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8185975d-2a18-4b30-9c3f-e1893c0fb4e0)

Current physical partition columns: []
No repartitioning is performed automatically in this notebook.


## 11. Check schema enforcement

Schema enforcement belongs mainly in `HP_07_NB_BronzeToSilver`, where columns are explicitly converted to the expected Silver data types.

This notebook only verifies the resulting Silver schema.


In [11]:
print(f"Schema for {TARGET_TABLE}")
spark.table(TARGET_TABLE).printSchema()


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 13, Finished, Available, Finished, False)

Schema for HP_02_LH_Health_Silver.clinical.Encounter
root
 |-- EncounterID: string (nullable = true)
 |-- EncounterDate: timestamp (nullable = true)
 |-- PatientID: string (nullable = true)
 |-- EncounterCode: string (nullable = true)
 |-- EncounterDescription: string (nullable = true)
 |-- ReasonCode: string (nullable = true)
 |-- ReasonDescription: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _source_entity: string (nullable = true)
 |-- _pipeline_run_id: string (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)
 |-- _silver_pipeline_run_id: string (nullable = true)



## 12. Schema evolution note

Schema evolution should be controlled, not automatic.

Example: if the Patient source later adds a legitimate `Language` column:

1. detect the schema change
2. review whether the new field should be accepted
3. update the Bronze-to-Silver transformation
4. intentionally evolve the Delta schema
5. test the pipeline

Do not enable `mergeSchema` everywhere by default.


In [12]:
# Example only - do not run automatically.
#
# new_df.write #     .format("delta") #     .mode("append") #     .option("mergeSchema", "true") #     .saveAsTable("HP_02_LH_Health_Silver.clinical.Patient")

print("Schema evolution is documented as a controlled process.")


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 14, Finished, Available, Finished, False)

Schema evolution is documented as a controlled process.


## 13. Review Delta history

Delta history provides evidence of maintenance and write operations.


In [13]:
history_df = spark.sql(
    f"DESCRIBE HISTORY {TARGET_TABLE}"
)

display(
    history_df.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    ).limit(20)
)


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 06c3ed53-342a-4ac2-a4af-8b9884515e5f)

## 14. Save the optimization result

The log stores the actual measurements from this run. These values can later be used in documentation and the Engineering Operations report.


In [14]:
optimization_end_time = datetime.now(timezone.utc).replace(tzinfo=None)

log_row = [(
    optimization_run_id,
    TARGET_TABLE,
    optimization_start_time,
    optimization_end_time,
    int(before_stats["numFiles"]),
    int(after_stats["numFiles"]),
    int(before_stats["sizeInBytes"]),
    int(after_stats["sizeInBytes"]),
    float(query_time_before),
    float(query_time_after),
    bool(optimize_applied),
    bool(vorder_applied),
    bool(vacuum_applied),
    "SUCCESS",
    "; ".join(notes)
)]

log_columns = [
    "OptimizationRunID",
    "TableName",
    "StartTime",
    "EndTime",
    "FilesBefore",
    "FilesAfter",
    "SizeBeforeBytes",
    "SizeAfterBytes",
    "QueryTimeBeforeSeconds",
    "QueryTimeAfterSeconds",
    "OptimizeApplied",
    "VOrderApplied",
    "VacuumApplied",
    "Status",
    "Notes"
]

log_df = spark.createDataFrame(log_row, log_columns)

(
    log_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        f"{SILVER_LAKEHOUSE}.ops.DeltaOptimizationRun"
    )
)

print("Optimization result logged successfully.")


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 16, Finished, Available, Finished, False)

Optimization result logged successfully.


## 15. Final summary

Use the real values printed below in `documentation/optimization.md`.

A faster second query is useful evidence, but query timing can be affected by caching and current Spark capacity. File-count reduction is usually the clearest evidence that compaction occurred.


In [15]:
print("HealthPulse Delta Optimization Summary")
print("======================================")
print("Table              :", TARGET_TABLE)
print("Run ID             :", optimization_run_id)
print("Files before       :", before_stats["numFiles"])
print("Files after        :", after_stats["numFiles"])
print(
    "Average file MB   :",
    round(before_stats["averageFileSizeBytes"] / (1024 * 1024), 2),
    "->",
    round(after_stats["averageFileSizeBytes"] / (1024 * 1024), 2)
)
print(f"Query before       : {query_time_before:.3f} sec")
print(f"Query after        : {query_time_after:.3f} sec")
print(f"Measured change    : {improvement_pct:.2f}%")
print("OPTIMIZE applied   :", optimize_applied)
print("V-Order applied    :", vorder_applied)
print("VACUUM applied     :", vacuum_applied)
print("Partition columns  :", after_stats["partitionColumns"])
print("Notes              :", "; ".join(notes))


StatementMeta(, abd9c13c-d6cc-4ebb-8884-d02aa7a5189b, 17, Finished, Available, Finished, False)

HealthPulse Delta Optimization Summary
Table              : HP_02_LH_Health_Silver.clinical.Encounter
Run ID             : 4e79c87d-fcf5-4a8a-a3f7-97d261220659
Files before       : 10
Files after        : 10
Average file MB   : 103.05 -> 103.05
Query before       : 4.079 sec
Query after        : 1.571 sec
Measured change    : 61.48%
OPTIMIZE applied   : True
V-Order applied    : True
VACUUM applied     : False
Partition columns  : []
Notes              : OPTIMIZE with V-Order executed; VACUUM DRY RUN executed
